# Stage 16B — Comparative HAR–TF motif scoring | no Google Drive

**Goal:** extend Stage 16 preliminary 10-HAR motif pilot to up to 180 screened HARs using the publicly shared human–chimp HAR sequence catalog and the named Pal Capture-HiC bait fragments.

**Publication caution:** a named Capture-HiC bait is a restriction fragment, not the exact HAR sequence. Coordinate overlap is a provisional mapping; catalog identifiers `HAR_...` are not Noble-screen identifiers. Motif scoring evaluates sequence preference only, not ChIP occupancy, enhancer activity, causal HAR–gene linkage or confirmed human-specific TF binding. Some loci will be flagged as unmapped/ambiguous; do not force them into the analysis.

This notebook contains the 180-screen label manifest, exact named human bait fragments, and analysis script directly. No personal Google Drive account or mounting required. It downloads a small *public* third-party dataset pinned to a reproducible GitHub commit (2026-06-24), and produces CSV data, QC logs and a ZIP. It never needs the multi-GB single-cell expression file.


In [ ]:
!pip -q install "biopython>=1.81" requests matplotlib numpy

## 1. Unpack the embedded reproducibility inputs

In [ ]:
import os,base64,zlib, pathlib, json, hashlib, requests, subprocess, zipfile, traceback, time
from pathlib import Path
ROOT=Path('/content/HAR_Stage16B_NoDrive'); ROOT.mkdir(exist_ok=True)
PAYLOADS = {'stage16_full180_no_drive.py': 'eNqlWtty2ziTvtdTYDIXINcUIzmHSaRhqjw5TbbWSXbsf+ev0qhYFAmJsCmSA1CWnFSq9h12n2sfYp9kv26QonyYJFN7Y5FAo9FofH2kHzx4cNYkKyXGT0VZDd9W1apQw1dGXynx68lvIq3WdWKSht7XVaOXojZqWehV3oSDwcfNotCpsNXGpEpoKxLR5NpkQ6xprsXx6Pip45I0SVGtxFobUxldroRVf25UmaqHlWnyCnM2HPzDKis2pcaMyFePngldNspcJYWorpQpklo0lSh0eYltbGqUKlXG7JNCJ5Ymk/1OptqGg/McImUVuL7/cC4Sa5VpBDbW4Embi2opIM5Kl9iDGJXJGsSVESDQWdIocf5GLHSZQeRw8ODBg4Fe15BYJGaFI1oViNReBeLCVmUg1kmTY6AqCpU2uiptIPLE5oVeBMKAtMY8XgZLU63FL7oKT6DHUrQsPybabLVVPKhMT3Wm/uxo8NhJUG7W9TXOJMq6G8L+dVE1tEX/GG6s8uTJaiX9u3RhfU1PxKYumsHg/E18+uH83Zuz6LM8+/DPYzmRpyej8eNH4RMZyI8n/3zqRkZPn4djjPz2+uy8pXn0LHyEEawaj93QMxDR0Mvzl286oufhMUbev3l34kae/jTCyJfB+a9g9euHf3t1Fnmj8NkowJ8n+PN85A8GmVpCf0kWQ9ceKdGfDMRWN7moalXySIDrrOiWIrlplsNnMijVFlBRkZQ+HW85MarZmBL4sY0HRuErnTa/gasy3tL3eZOt0Y3a7xIAQzbImcRG76tS0b6wgBIao7mJcDzvCCO38jsEGgixjTpJfqetIUmw1KrIGIdRuzXhkaWmPWejuR+oXWMSmzDIIgm4VEZJf7oN+QBumbd/p2W81u81uVDZXpPVpok+fxmIJe0Dm/uKVom800AR2sbo2vNZPrzA5C0pwpM/gjCtykaXG4UFdVSEpiUObV3oxpN/NMAj8ypoM//nx4crINKsnj2az4HDNIchyEmNkweSd5ETOAavno2hCqnKrHs/pnedEe2jOQ7U3jiY9QdfJmCxP7pRaWUyi+NPL/mGp4vNMprNW2Xgur5DH3yr9KdTyIGOMHp4LjquI+yV9cJxobnLSSfR7HIOoIQXlS49iOSHm7qmOyXCS95sNp7MW2X6UM1ecKEKqyZ4CxMsKTOPiP2BY/9N7q3KWjpnekUFvdXb9V5r8B+sMrFNyiayqvH2fiOE49wo6/n+lPa10Vd0F9KFdIhgamBWRyOYU64LaP5nggZPdLjjl5me31Ag7yxaLa51FvVUrXrG81bBmKULJbmZWlxE+mjMT27Pi4M9RVJm+1u0s4ubuxaqgWkNa1MtkoXGPtekF6N3E/mNlXt5L44itzcku7Px397UHUisIwN8qcSkuWfkNvrD/ov3R3bky6DjCMcQkcGsw5WpNrU39n3WDSNHjBybxJhoxl5nndTeEhhoOgZHfx6N5+FNe/bBg0zmT1KvScqV8rb+3LFqbRwc/Sja8uGSovBobIeRx7xwRwuJZMLwmuGm5hHeWxbRxdGWIMAaA0MCHdH5P0R8mSZB5BT/QeB7TSkGtKytpUSj1iWlCf96cvYRAd6lMO9ewXfLIxzBo9XDPTe/NwF6d/gH9GObJqWHnIVjAqkaucDrHWInUqPKXGNR4XKkj7+fIjepKNwTAN68O/0g6iFbBfnJNpcQVZpu6qRMr5FEiVpGwquPRuFoNPYfemN+euxPKVVBYrJI0ku6KegNwfJJKH6pEG0gPDSJrEm8TtLuVVqEojKDjIIkxsGn4v1+KNMYNJnKQiQbuMwMjpWyN8QIi/NeI3XZlI1otjpVIR8Rsm2jsg6RUR17Hh4QdYxJrjmcBFlzXauIseF30o9J9IckJ1RZVFG9DdfwMslO22gMtGzWMPJc83iyuzUOzwZQuFAlFso20ZByqlCXy2kKV/O5mYwYLQ2hpc8Z2uBlaZRviW4WuRI9k5tB9mgR15HKFmqtSoYrmQst0j1iaXPrDy+Pxs6YdhG8yEQfXXbuIymvvYTvFWvkycu355J5JPS+uxHzHAgiOhcryKu3s4vArcF5MrXzUn/ujOYiSImBQkqngCcFs2BHD1BFHvMZFhWAkWv6pRnWDemPHgLQ8eg9mpl0/gU0L6JmAi3OmjlbUYtzZsH2z5q2S10CDzzqO4/AmU+Alc4cEspPSYsxoRMuyEuCRWsSLqFdED6RV4MmafNgtxkhf28pOsM8ebCqdDwV+cxyuEKejxx6sy4J3O8rUefXwB+S/QZghc0ownoBxCeE28RQjF4qY4BrhmxSlNGtZBrYwmi4rjLkX6sC3rOQbiRp0jx2d3XsRrQ9HByOeZTCWAzJutHHPIo8DBH2cBx6XW6ziCb5TKwbCs8mjTpQLv4CklOM3lhpUlo6EGleVVbBr1+5m7wKebcX2Mk9uXvC60A05pruPK0itypkm6bICnVX6yitwlbxWtkjvHXndW84ih2IPTbkXUElyeB4U8FHQvHuEujbwrvIgF2Vd2Ojh7x58IQDDT8fQGsg1C5VdSNe8w8AM/n/iuD4MlzXCdwPwbOOuqItPDGrDTH6SG8EDji2LIuTdtiTw6Eu602Ds1C1CGRF52aj7qNbJPp7yFy5+h2ESFbv2XhABaeN6pDlpwV0oxagamvK8CN+PRoPWXJcNzzE3UnHnmfD9WWmDXIzg60t74PCAhE/ri67XVEmxHm0rxiw4UOZQ4SYyvMQQ6g6mCi9jwgB7txUT1u6ZdKxcjm4o9uskzIm6pBHHV16hy7NUbvepENYttE+N2WqNeARL5HHhc2O64u25CfHzkfxX0SPjkcjl2C1g+ntQRL0nrGODnXwja5HljQJEgg4IYfQRknE9B/Fm5Oz8xPR1XCc/cJQd6jaxAiwsXB3eVIsh+RaBBc5la3WCqaL1FuX5Nzg/wjCfOIY4CZ5A34LrnTGQWoRYWyGtzmi5/LB58WsLZjmXyb04gqm+ZchvVCtNP/yYGojZhKukPdc+r2xWU6t9oe2yM66VcOeV2+6AMg1XUp00PMIIXGyKZoMRa1HCaTvgrPhONtV8QxGsp0uszeQG9ETHlXOf4hk/v7spZWUMWGCbnG9KK7dDJAnDyOtE2EGOmdk8a8nv8n5vCt/PLPXSEBZr2nPEY/4DiRqxnaYXDk8UbGxCE8YpyD8o3ilsg1S3BT3MRFLk6zIUjlbWiicp1aYQEK7RF5OzTIkeBTluCPFF9hq6PPlhLNpSjWv2mT5MrgipTiKEFF3DatGLuPOEd1UlhuEuSF3RkqOWm8q6oRdBD1yYruvXJFCEWe3hjRMDYUIozeVNKXz2KgVgNBAdMFsTphA/phxD8y6upL4AmcBXyRbUycyJxnVFSc7lJSMAkr5TIucQPlDGjV7AAW2VUAa2IArbJaDvXoU9fflt2lXdfVi5BIZ0KVRdfVwz3zYc2XqA6n3ACChq6uA1nJepRBe9aJQ0ewTS/GJJOjX0Y6fZsfzF1H4nKKvIJxEnw8VN2E1SeRrC2VieoE2gNiYEB13ILFywq6DzuYHEslj3PYxUZOs4rZTGbe1tiPuxcCK/UtHgxgIBrT47evnozpt3KLuQISdrvPAu05YdtZQs7ESRX9Z3RaX8iACVKcuWuFPb/UsDjf5IRpPbrNNgPvVpsLLHfIX4zY4Y+vuyMrFcXccCNOd5luiHNxk1G3AvQ8QhpuaVOV9lt0uGpHBVFfawi0h35vQatneQLyo5QSs9u/Ek/yXnLj0hd6Rr3xpmzlY2+X9HDPuaKDb1AWqQi8b4hxvyjUdJvvm0fLWJ3duPs4DikPs56fp7cmUJlOe7K88504hPaX2jnxdvz3G5ayaPK5M7EIToqwuNkZ9U8KDhD7gzC66WwnkNkhtKxLTUH5GkYIk44GfR+FPd4TrvgHEPceuOkBKUVTb+CfCBwm9KZMrCJzg5r8p8e1tagokN0CxRySnwHLqVhycFMsO3tp5V0DcIyyo+fE+wcij1ve6ka/iNQeijJwwIjjM99EMc2079HC287GY5dbo4ZzzxzLteaa3eaaHPNPbPNOeZ3qTp0vkWnCRcZETyNmLOYu4OZPSjFvz9uUd46NAwg0h+fal7NtDYPewY/t4z/n716ft+rRd31/g4Y1PDoH+lbue8FMg+SNVjQyqXS4PLnH/OapDOIfsKRup2tXw5cSTejxtX6j97kSMvkxdeO9QlNZ+G4WbZdA2Mvu+62EotnmQ5tG+bQWjpFyZW2ooGtIgTfvJ9HCSFn/6+xg9fyMnkElyDiIna8Yt3y11EeKu4nc18v6O8rYctDk5CdIIO4p9ctnd79d4pB2P9K94ZKqA33TSICvZ2PjbXPNhz/imcJQZ/9VeX/YtmNzc14SBz1IFNwFAQC2AukhS5clQBrJGPfNptuzKId3Y+DPTf4FLSfMZVsyZoK2DbhGkPUF73ns5DFtCvuaZ5K4JxQTyjeovmplT7jVxj5CTXjo3Et8c1QrwgIg/7XuZAtFOJRbBJJPciyc8dPD9BHT139dQfz6UZ+P4pCjGz0bxWRedTtvk6GST6SYEpQycL71n9XHsujtZ/KGLH3s+//7SLXYmdM/iR/GZ3pH5kIzxGd1//JK/dWtLVQyv5gNgsQshN6qcl9RWUcb71AeYPpnci0zfhcNss66RlpRx983aucA2oYfZtGd2fGLXsJETLqDcGIjK2B0l3n8vd1zaAxIBSxun/SEchTtEIF0HvKWilorzVsej48ftd3mqY5quYX74HR8819TmkIXGr+vpyQkCIXVyUAkbaslp1oygnJIMZGEqSi/AJynp2zqh3NXNnU9898p17whS+6/wYIgLBP4oJUGp/fsp2ZOyODNMT5UAor4i9xtIAubQqBXt6vr6rhnGudAenw66vVawsKxEBq2hED9/c4BeKkiSjYUQCHcohg6d8fxLwF+yHHjOT87/cRZeOG5bSRUkJ0TH7jsbC0NGn7vvuLdr074edrc3nzZLR8kl4t6pkz8+pfb7clMUHrenHUc/oOd2ke8HICmTsgsPzNnJcDpzC9qO8y05/KBl0U/Dlc/9OdVgf8dxkkfRO5KUSlUg1HMSUdlHgwvrnfpB2+v3Z5PJcDyfTY6fzKen0elM7+ZTdlXQlBN3pud9bx7T9A25aEKTfkxMsrZ9qr9E1WFUuIQjhXFWSGrkFr6PkJPslL13YokEMbT6E/z9c07vwQSyRbSD3Szo/yGshzEiibznwdPwie9P1zvut4ejJ0F/NncwzOp1lOxCvQZMt95pkFKBLn8xv7yVwRWUFw3XOzxgE/wmtgb0UDNtmgpuH+usauLrRqeX1us/RDiN4HLbh45wd5uwh0H31JEiTSkQYA6/U/F9/u9//hffZf+xSqQ58bPCa6paHD8RRZVq+mRYpZEs1BK2T2rbKvq/n0guYIzUqdOrkLW7SIyn16TEZOfkjeT//LeLHGVl1rCjT3Ap++3atQ1xi4vkuto0nhuyyZXCr7OzN+PeoX8oi+sbzvqs9wVhXa5kkNU6Qizxp3SRaVFZRbeIC64NNXhaq53IYO+UrXfoY3sfKoPeb/qDASw65qo5jqNIxjF1lOMYeQ53lgf/B6ZMzUk=', 'S1_All180_HAR_Sequence_Mapping_Readiness.csv': 'eNrFXM1u20YQvhfoO+gBhJY7+4+eAgNBTgGq5E7QEmMToCnXktzk1nfoG/ZJSpGydhytVqeZgRDZIG3vl9mZb76ZWWq3fmnbof70YbUc/9Xfmqeu/7HczVeH+uHQbdrd8qk7fVvfN/v1Y71u+3633P+9PV1db5+e+3bfLp+b3a7d1ePPQ4V/A92w7288Hp6aof785a4emqd2M17t9nU37NuX12ZcY749tIeX7ZD/ifVj9/R8/Q80fV/rEeB44fu+bvquOeJoXpuub+77EXHXb/f1Yej+OoyQxt/fvrYvffO8bF9HkMO6rXf7Zn/Y/foLfB8t9JtSbjl9t4SlMcuvL4d2fvs4rtYu1fRCF+b/6593dTNsjgi74aHeDv2PPxbDdjEi7DbNvt0svn5c3HfDZry7+H0x/vX//vn3oR3axUvbN/tuO+weu+czBAhnCAAyEPQZgXUIwfSGAExvJOtDAcB4j8EC9oxAa5lNcAmCEYLgS0aA6UWNAEGoLiDo8UUOIVRnCOrdYvO7GV+aHEOKSO2EzJCY0cLPMWkmDLQIoLrlCtTxAJgY48820Bw2UIkUVHXpjBysAJCsMDKEDAZtEoaYD0pDjiHtBViRoASdtILOyxWgTdVgSjbgSBFgEi+YSgaCTdyohaxgg3SuBo9i0l7GJAs/BqQXoogZ9A3pSr5+VaKE4/qaHAFKUlomQWiVzJDJDywbgdSCkSnkNHJGf+mMit4GuJILhVKSEEGygTMSmk0bVdRsLL5oXMkXWcxgkyuAkaFmVE1ClInIgKr6XCXHQM8xhaS5LOTohbNBxJirqI+7UL27QgKiZAYOVzDl+oEHQgoIbwtqhap8MIiXXCiwMxkAn8gZ/MUmsIQD0qzZGkqRU4KJJVKqphd1PEZ1q8VFb4WUoZyVkAq2Km6EYlBstrgNHFnaIukOMp5gtZJu/1uN+q1KwhkdSM8f8ATEyyBI1OxEqnmPmlsy/YwA4sU8SpC5Eo5eMsdEy1AxO+KnD3efv4yUOH2VKFxOAGxC4LMSQdFJhBkCIAjcXdYTgnhG4DzvXHoGYBIACCJ+kPYgN/vhsIFFEICbkGYI3qRtqJiL+BlBCGcEvuJNTDOA6JIJVMYEih5BMkGmn0Q7lp8WBsSJ14axxEY4DgCv+wFp/XxaP940QUWcF+BMCOqolPFiKTlpchDIENmBgyZ3R598IXN2jcUKMUVERrFreiNoJBK04W5lzBD0GYHVvIXbaX1IJtAim6BTcuCeucwAXAoFDRI66ThquG4C2rbiDABlx1wniZ4MdEiBYGRSg4FUN2mRksFAMoLnnv2dELiyWCWPRoNEAnCfXpwRGChlBYbkbKwp1c8MjGRsYiSlROom40wRAocvBsQIeYVCTMwmpn2wFe/gcVrXViVPZNgDm4JRORFHtJAgXE56OAAYU2rocWwCaiV5XWiqElWOdizVChKNITFah4Q6SDiB87fyIrUT+HiTkKl3AanE3HyBwwohBePlsX6g7+LYkBxBOwlxYFE/8XLaCPQNTVchE+QnHNQItJWlA4eqZtBZkUqNwOhC0cpAyc66sk5nQJAY0YgkZpQULkfOLOurYk4ADgiJkI1I2e4CFES6ZghFlJPYDweeECA6guyxNOr2jYumNOJhGDa68pCJwRN9pUtZQTEM3hEhWCuQFbxCwWgEgtGrKNvP9qhelWEDr01JnzGIZI/0Wb5QIBep3kpX7d7dKpqp/cCpwnSLwxEd6h4F7id/TxB0sYNHP2L0PhnBaIkupkd9VO7HJ04AUigYkRGfR9rAKAkEAR/KCyKH8oKO0lOFgDqJmaNIhj4YghX3BIfCMQhotOCK5y44TOCLkx0OG2BOtAIyMVbFSGDYhFjFchuPnpEiOqIqc/YjginRMoMnRnwKKkpM3CP4QjeVwwToAEzu0yg4/ODGEVnyeiUirawrEUfEE/8gcv4lorNg2TPzDJTkrbgvelduI9HvA2qsO+YHbccbqjoaYHX1zAHQFm3j+uG0fsYJqQ+rr46fa3ntWXdqYbKan1pZiWSClZp6NjKfVTYt/2Z4rdgbRis1lQUrmafGVmqqCUQ+x3RaPRYCnt70XtLyU39oJfSE1EoF9eb0oVSPUxwtGhc3kpaPbxGX0d30SS6ek0y+BqfNcVC9uZ0JWb0LhPsOb2STfzKTcmnni0sTHqRbRb28/slStCT3P7MPRcw=', 'S2b_Human_NSC_Bait_Fragments_Embedded_for_Colab.csv': 'eNrNXUuPHbeO3g8wf6LXhUAPiiKXRrLIYpABfGcxu0bb6es04EfQ3ZnJ/PshqZLqpFXVvinpOFkk9SWmz0ep9CAlkvX0/vH+/vPtj2/eLu+/fH6+//15uXt6uv/07uP/Le9/efzyaXl6vnt8vnXv7p7uf17uP/98e//7+4+/PT38z/3y7u7h+fbx7n9v7z4+3D3JD3x5/Pnh893z/dPtp/u7zw+fP/z7v4Xf5be/8x6XX376x/dPyy8fIukve7/I//Q5Rh8LAkpp2cTf3/36/Nvj/e2PD9/fPt4/PT8+vH9++PL59p+Pdx8+3X9+vv3pP//r9tfH+/cPT/eqf6MK9JIKlxwCBrAnBfTLzYXsfzx8Fijgt093n797uv+4fHm8+/xB0c15LeILJYJbYqbkYmIDIWZsasTlxzff//QPn0cIw8tmu0X4IKVsT5L/vrmQLc2OMW0NvwI7UdInxgDXZE/96KKYEEPUJ0G46Osknf02j3Q1vqTLi48UIWQwkCL4ZZM9zZN3BjIEl9npMyRpys0mWroUfZjSpfvcwsn6BGC6IndHvkQKwWv/GmC/TeHc6OXPLybx+18ePv363Yd377fpvHx4ePfui/3x7dP9+ytoyD4ZEE3joIbztROlsoEYIo723/P94/OAiuReqihTRxcrz8wFhZRoaeL6wq5AFxKGFWXHG901qLLudIqiE6FrUglBRbK/XnSivrUBvm5Tk507sgdZkAxcdmEcfGMHXNaJAmQ/v+CazhMwrjw5+4s2DXbgSysohAVQtifObCDJ9tFmpkibWRBkhZX/IQqtExUwXkzUD18eHz5+vJuwqh2plzysIAY/oN41VIshFSDz62+mWpKpZ4BkWgy81LGlNridAU6yQmSMBTG4NuhE2tRLquZbz/rvdfH94U3gNDbCjnRh8AV5sY/j67oUPabroMy5IO9S+Ep/2DJQOmTw7XSOwn+LCSkWlrwUXBGK9dqkh1bVYzaMzlB02vhNejaT/j6uSK2QrV1DK2vwqfc/cvSssyrH4FJqFo/J2kR792FkLIfY774xInpHbEAsrm1ehTiFNEJn2nmO6NVjNiA9uzTZscFyxCXuawGYeOOaz4O6QClgxGvycHYGood00XdjA3LPJQ1BLCVwwYBHv7GlwTd1QCYcXADL+NuEpxMF53IBCf1Fqwb7kPqlOninawcUlJC2KSYytlbDwPyCHc8ekIFhRWJwbCsJNM/38vholpd5pExCTAXpkP3XlRncqWBn4wSALAq5grIY5Zs6684pOpatEzY/0+0qeAXdRCMsSOYC/zW6pe7o01yMhA7lTRnyeqyyic+n8gliQcHFuFGNLTupO2lNi6zV0tliuygghm1AJNo62c89bT3URBQomnAC+iaa5H6TCRAhp6Q+vyJ0vnkqJl7NgsHV4ogZXagob2fOf2C+BithLijLeNhv7+CCRC+PucTyiZTFmfYG4HLKU16nvG4Sb3WW1xGA+27XlNFwoCL4HAtI4P9aFfvjD9m3Seawh4JSSq0X6z2F6VfVuzjCX9GHd++nKyQGfCwInfPfVCHXvUNPavuCKwhdpLbMRWcuYmovL9HuoeqUs4cD1dAl6SJDssDRX6Ra6g+hZdg75/QJRHHTy65kmAbYfHcUSksWa9sxRwMhXgxjX716bscvKez1xRU0Eq931Qi3LenbaLQztRyIGuKxFiS23Da1fNMEA82414lx505J7LIkL8mATqRG39g99nezNkwnGGdHKqH0hoHsHHxrlXauUn1IXgaO2v2KgJo7p+LnqWK/fgRKSbaiWNDFVYVID5mLh2zltsKQ+DYb23wmcNaXglJguGjXkJ+qt7071ic4dhwKSL5d1aq0TfUYZpl+xwok8f8NZIZXFRghx73Ld/Ta1/rUC5SbTbZs2NBuHcVc2+bSaD8cqEIyrOQZYPNPv6rKfDWEvqghi8q3UCO9XEjUD3ApxwwFQMJtVIh0OT0BdwVOoSIDyW93z5M4u+tuWSWRiFBjPQzlGPKySZ9moj7wIyS1asAVwNjWZRo70zvkYixA2uc2ruk8xC4a4Eh40aaxlZJ7C0Q2NJfFXFUQfNyMNF4vyjZHKDBd497zQKvgIRaQ4JRW8zWSHdJAdBcT96/USBShAjTO4MybGzsDgL04O9HKxSxboIKktzGb8Hki3rPKoixngWJBIEv7somfpuoO2c1OdyRbBBUkBmCd+zBwnA9xL8orMoiTrc8kC3Z9qSa7Ru2JJ9WdkQ65JkeKiAKsT8wev40ie1u3j+KjBPGOVkSxnd+ZfPWm/Sw75hUtCFYU9cblK1pcQ4PoGgJMV9Qg+50Jp7f0jmMB6WJ05tWmlb1ksO8PeYW4AL2O3OG9Aie6XNqK4eJtX5kzWGi2gLTfztEluzsTlSZSTjk4O09T5FM1o0R6yIw6ZtMDKEPRDmMb25CBA+z7GxgWzz1EMeoViL/e2NiPte2ILKq5q0D8XdrI5hOBV0dTQcLLVg32YXcAsgCEHDInA7IvNOvHhMvCw4zTFuADFYSZDMDFvfORCvPpAeUPDLD/eg+MTVNd7f6og6x+KHYnUjIgrlxbkkx4Xf4tOn7mlVryO6YnEwbtDUMxtWOj5Kfz6E3tijJB4xmauqk7HZZOFSNHDB1Z6AuSDm5k7kpsKI02RMmHjW0+k/w+GspOb9+3dg0tFRph2a3uHMWd1EBVQ1E8gGWTPs+EHRNm4hzF2zFAW+CiCpdzDdcCC2T1mhxYcKgSycRUQOqA/S1UEk3AAPvt7vvbqIRhL/GMWJz6wAWJoVNnGp4/TO+SgnQE6sYIUEBO2+0Gtks49lO2DMz9bZOe4ntN1FEQQgvWwzydRryjWABybjRDi1beWUkQKAaHvKKLk+W83p/li+tN8N0F2lAfH2qU84rEU4VvqRGFPgQ1kA+cZR4JiJaQdNOEy0kzbhk/MezdYw0olPdmW5IO0twjQ6hx5zdNvHgV0CJywO2aD2P23KFaFqViSLZ2/vNqXUEl20gM0YXT9yd6aszuY967EQWfkxjdCsjMrir752nKGHR9KK7HKP/QimLQq+gmXJqcI48tluX3Qn/kHcWtjxQK0HycpYlOZoHovIEEwBNYuHfNdKcNUZZmBVFv92+q8JrJ4ctd7s10WnmFK9A7t0ZbKM/TQUeXF50q3rloIHB7ZXB+VO5EN/uQPYSUsSCxK6E2q63rFGcuWa+rEjWgpyDNl//XVBlQI/U3y8gke510swLGDEuTPUuT+1Dn7FLSE88CZEerOfJnsxpeZ2JZeBTY5WwTncwSxAY0AM7R1p6TPlD5+9SFvlqWqYwPVsNTEYiFUkcKtf1DLxKnjpVDTSDQqglq+Nf1NWHsb4VkeHpRJRoIKdRRyzg0no6ohIEMWARNk51NU2JnFGjITmvR0Ijil+9R1j6XzYdPK0Ks05FprPuOuDD7FYnB0bjm81ikkyIGuiIPA1cePUNsfTfyokIXYomLev8O5Q8UiP9ft/5QYiy996dn1W46jpqpkGV0GkoQ12mlwrN55NdpRSnhFXn0wK8grhEqQzzcr4spaKpUJgOYqBoVKiy2EsD515R4JxdGDAa0i2pFMgrra0o8Mn0PubKltigiCKFxnebh1Heh7qHIsp8oyI6rMaTC8lfTmIV7SJk1WtVATv4l5RXoxPEqgHJHd9rXK/F/O1GS6L3TihjyZKg9GqOtHQ5W7yHyWN8eEMsaqU8ZpvElcSGdTGgX+V6jxNr021q6ZnhrUwc7OfSnRuJO+xzVQFDE3hbnVXg2j/w6r0hLQV2PRzM/C+KaXyfCI8tL7I6TdVJoxTDNdFMgu6pvr66FIOg+MXwT+LoGQpwMyNjJX9FgPnsSU8wAhui+1v6x4Yu0k8YgizqhTBoBMTgZETdNWP4qhPN0XSwnL3q7TBj1iSnVkZXz0Mja58HETp/ZYWo8kzmyznZ9QvRX49CoMHlSaDM+5yFbM+74dsxiujgZ/goy0GpTnLeUYleMRxP5soOQtDSgItBM2Zsm/cMbyKcHG4SdFB308obE7isgh/Udqex0mhwKTUCmCTQvO09Y9AJUX4wCLa61dp3JrlcAefdOYlpJsld1A32vClIk/st0wz7N3nnxobS8hSF07aoy4HQeYfIr0pqQ1+OJ5ArK3tE4T+fW6EFE8IwWKGeIXZ0+ccjPOOZiWUwNZeTYuKbzZL3WNETSf1ubRpZUgNCVyIneE3kXC4hQmwRhOkvUYoB6/w1t0EEYa1CC3huUxVQzsmNBEOvBkQqf5sk7OcLoM+sloyJynOsIT6c3V0DoVq3gyFO2pBYS0xprAgsOnUofMYl1mwtIMTWm6SxWfUIB88YyNhT60oRu0cAUWYOiAWhOD5wu7vg6FajjY2BrFrn5NFqwUkHWyOjWoqHe473LYS0pFZPWKlIE0M72oOWLnUzkWyN7d/KASOw7rQemgOqZWxp6X0c8ls4kQFaJWHkmc+iZXgGZWjzz0Jvq0t30esxpgh0UkHLNhknhPMlODBM45KD3TAqSlrFssmdpAPpoEtnuSFYGb0CmbD0oMeEa8Ulpkrt9qIIwUwEU6GsqXIGexMdVoCbHV3tgyOFO3VW3FnbjLNZoNoBaK7GJTmaRH1+BeHTXY9FTGwUUai7yCEsXuMsLOasNFg2gxt7fNNkf3qQBp/GIDVFcGQV6sjePDXcKFJDzwWnBRgXSg21Oop1exhYMEP4Qij3BV0p9/J/eZ3MireSugDUKoMnOppFf5wKoXqJdhYZQww5YNqU8oTWZ+xNTTUkXB2JFSa30JjybJ2l2VkFYzeKr8Mj6uCKqxt0IT38a5BYU4yeLoW9AzPxaNuV0uenXqYQBCqB6jZVOVZv+Cg2JH66AES9aNGSZEOzcRPugJSHyijI1MhjrviOubHXIBYnBmhrXeZ7c8xAyarirAo1PXJroaZadYnJZAxL0eEaB+Ep1GhFNpwlaNURBdD43mpH3g65PgksQksaeyzOLQ7s0wbMcfWnQRfYovSZAA8w1NgBjOs+C3e4bUKyyqEGpGIJzgZcmOpdFflxrJyoINIEF4s6ZOjAgy7JmJUdTjRBV2dk04puQAeL2/gdo0k6EUmRHIYnno0CssHUJwITTaaKW0FcA0fEEmt5yDcGLv6WlshRoZd2bJquXxiWU4mY6pX7ERgFBi+LbKKfTkZ4m6LdWnJUne9nCId+mt7fEveKsVYNdARjqFQXm8epIr5OqtSIgOZ0HPelsQuGphOTjXisH+9b3VlmMmgOc3Ioi1xU4+6H95JArMq2IkRrXeR7YrWdL5ACoIN9i9fF8vO4xkddofUMB6l3gCFGfsaOFLlIktcUUsAv1Rt+E58yBI1phAwXgnMc92umUwhRcAZqAt9fSsXnQ2aJ6dqrRkN4VkH2bBjTwInGnSIvTqB6tJmAIWny+Sv/wJjq8mc+XeEUAkefx8c6MkGGq2bvOQNSslZsq3PJKwthAPaKNwa+0qcXbqLBRDtDRbhZW0C/GYUGpXVTg+bjgY6IUbKnRkos5YCMaWZ6z68MoyDtpkVhGCojaSDHZmnu7m08249yo2y808i2CJiu7gsTqra9Vheu5asQJx7pH7KxVTRTJdsLheuxhJ/iQsowwXpFumEsVHnrzR1xRK9QZglgPMlR4Ng/o1xEMpZB4a9PISUb2O0HGOSVZ5DSbQxG3dBoRHuvAAy52viK9uW7C03kgkaHsKIWtTUMd2N0faYJpzvpdTFeQJQw34bM83QdW4pJRD1PFRFOggU5LE53MknXjNaCnGOMsaSdDVb92oRckCsSxrSxpaMgdMZHVJwBbnRrTbBZylm0sINQST/n8d1vK3+/CQPSYSawvjWwxwL6GnIjsUN8dUbF+G9FAqife+XzAyTENyggTEFyuHwrTFg31Xn/0JL64LwWZhZHaNokTrPQDNmbNNrVaYHPZfB8JmZLTT0RkA+xqFJfKnqYJO8fRDhKq5V8Q1HP2jGE+j0baFJTb3L0Gj6aTFsRXbQ/X9mSqhQiGeOJeoqhsE1FPIg1prubSpM8S5dCnjaOmkWmSrwGugSAiO7QSHVIxJQPs6426ys6mYR99AQkn0Ox9WTaChkGI56AAXI0GyudjqnLn1Op6IIsqU9Qnuuqpq+RZkt71WoJHaUvUD7ApAq61UfJ5F4/6ND8rlJlZ618VRL4eVtH5IFXaKVSWSEvMa16hAg41JEBl9eQ2nPeRD+k4kIEgTuz16YJeuRjYwtivSYdQQJTlaCbd7jkAc+ZkHwNjGSOxpk+p9FkidD1RjOz0s4oFUA1eVdnpNLTS5BaaNkSTdooRYkikZ9sC7JKkiZ5l6Qo86bZE+sXSDAXJ0rQueyo8mwf1bK0gWZUaz8i+RLRTtEorJ0LQJGdDUD11oitx6VdYDaWAsXFN50lWo9lQu+qgwVhcTWHo4+f1MhucAcj17EFFT7Pw3rWmTNaYCsgtWENlz9L4tJPyrie5iCsSW3z1a0V4ZDRw9/lKjY1F75LF1Cig+sVOlZ1OQ7KC29d4NTZgnKbvOdSBpnmgChCYlyY6mQWTiwXkQNdjybGALObkBJbdIHYv+1vQkw1FEGuEi0pPJwKrAmoI6gnrCFHc2byj7nBaiSnqGUCs7gSfz4LsSr3Yd6UjsPjHBeSazzBWu+aQKLN+wFpWHk3prqJzSfS3UwGtcM1g3Rru3FdZQwFDgCwmiILk6imNyI713AFV0vtUA/r9iSZ7mgb6j6g7ZrEQyICHao7w+bSgIxafzBphre89g4X6MzSUBSBoRQEFEdsKiucnT047Z91EQTaEXFDINXhLhU/zYN8cr8V3dC1QoFvD0mTP0uxUYwz6CUEt4mkgtQ1h6OLjkCjFUICmQFTR2SRsHyAMMYd8NRL57RXoDXLtstMLzlv1C7vDi5iz148OG5DVcy0ngmcpaC+GAjRtIhaQ7ZJIJU+//VdocloBWXK/SU6mICIDshCE61CIjeEKKKatddbAe/ddJEZYIAiLhtEqoOBKb3kceClHNKRFvQyUECuTnEzBa0u4xFJaSwZ6rEsDsuKdPgHqMyCUaXIyBejo9+V37WkROUO/v5PXHMS/0BqrqaC45t74k5dM8hexD1KwGmSyOBbEzu41VXRgWB3xsKsFiLmFe+JsDsRS2ljj90Jty8DAwp08T4TgNDRJgTzLjD959PTW447DDFGMCP3EvSHQdP1V9CTHXiyAVlvUj6sZ0iwvPYQssmuRH/7aR8GHI3eOVROF0oqA4e+lmla8L4hLqNg3V63PUkkeAun3qhRwMqtd5U7+/u43DdDp5/hcQYCWGmWy50jI75GA+M86qQyRp0JCfmBFOiSSn4cVpXWLID+dJFlhekWI4VokaNGEgrhcaVmXDSx8XZCrfkBbdiEZZQaoGNImeI6Bd4oWgUwprxnCBpDKdsRu4OUf0uiXyBWEaoKwm0wR9AvnBlL0V6LQsh0GuLVi5LV3V5nqcQRNDgcDlFMZXSevMd+Kf7R3XJPkd1MB6CxSTCXPv/ZDGnQUC0h2U2qSkymyKyBH82itJeffSXeWqt+w8zE5/WxuQTlZtK2Jll0I035hq+Gt51AbjaIw5MXS+1ba7OZXiwKsnxg0wJat9/ZkcvUxAUcF5ByUusd4vlrgW447VXO0VIK4lQrEwTAOjtM5MqghqyDbd8JVburvE5Q2cKy/f3Yq/D8fyjnK'}
for filename,b64 in PAYLOADS.items():
    data=zlib.decompress(base64.b64decode(b64))
    (ROOT/filename).write_bytes(data)
    print(filename, len(data), 'bytes, sha256:',hashlib.sha256(data).hexdigest()[:16])


## 2. Fetch pinned public BED / FASTA / JASPAR2024 MEME files

In [ ]:
SOURCE_DIR=ROOT/'public_source';SOURCE_DIR.mkdir(exist_ok=True)
RESULTS=ROOT/'results';RESULTS.mkdir(exist_ok=True)
COMMIT='a9251f39976bbc15df1d8f0dfedaf9786819c298'
BASE=f'https://raw.githubusercontent.com/athenamarou/HAR-TFBS-Project/{COMMIT}/data/'
SOURCES={'hars_hg38.bed':105973,'hars_panTro6.bed':105765,'human_hars.fasta':945122,'chimp_hars.fasta':944312,'meme_file.txt':552150}
source_status=[];errors=[]
for name,expected_min_bytes in SOURCES.items():
    dst=SOURCE_DIR/name
    try:
        if not dst.exists() or dst.stat().st_size < expected_min_bytes*0.9:
            result=requests.get(BASE+name,timeout=(25,120))
            result.raise_for_status()
            if len(result.content)<expected_min_bytes*0.9:
                raise ValueError('Unexpected small response; check public repository')
            dst.write_bytes(result.content)
        record={'name':name,'bytes':dst.stat().st_size,'sha256':hashlib.sha256(dst.read_bytes()).hexdigest(),'source':BASE+name,'status':'OK'}
        print(name,record['bytes'],'bytes downloaded/verified')
    except Exception as exc:
        record={'name':name,'status':'ERROR','detail':str(exc)};errors.append(name)
        print('FAILED:',name,exc)
    source_status.append(record)
(RESULTS/'SOURCE_CHECKSUMS.json').write_text(json.dumps(source_status,indent=2))
print('Source errors:',errors)


## 3. Strict mapping, alignment and six-TF PWM scoring

In [ ]:
if errors:
    print('One or more source files could not be obtained. No motif statistics will be claimed. The final diagnostic ZIP will still be generated.')
    run_status={'status':'PUBLIC_SOURCE_DOWNLOAD_FAILED','missing_sources':errors}
else:
    command=['python',str(ROOT/'stage16_full180_no_drive.py'),'--input',str(SOURCE_DIR),'--bait',str(ROOT/'S2b_Human_NSC_Bait_Fragments_Embedded_for_Colab.csv'),'--screen',str(ROOT/'S1_All180_HAR_Sequence_Mapping_Readiness.csv'),'--output',str(RESULTS)]
    print('Running:', ' '.join(command))
    result=subprocess.run(command,capture_output=True,text=True)
    (RESULTS/'RUN_stdout.log').write_text(result.stdout)
    (RESULTS/'RUN_stderr.log').write_text(result.stderr)
    print('Exit code:',result.returncode)
    print(result.stdout[-2000:])
    print(result.stderr[-2500:])
    run_status={'status':'SUCCESS' if result.returncode==0 else 'RUN_FAILED','exit_code':result.returncode}
(RESULTS/'RUN_STATUS.json').write_text(json.dumps(run_status,indent=2))


## 4. Inspect QC; export output ZIP (small files only)

In [ ]:
if (RESULTS/'STATUS.json').exists():
    stats=json.loads((RESULTS/'STATUS.json').read_text())
    print(json.dumps(stats,indent=2))
else:
    print('Main STATUS.json unavailable. Review diagnostic logs in ZIP.')
zip_path=ROOT/'HAR_Stage16B_Ortholog_PWM_Results.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for file in RESULTS.rglob('*'):
        if file.is_file():z.write(file,arcname=file.relative_to(ROOT))
    for file in [ROOT/'stage16_full180_no_drive.py',ROOT/'S1_All180_HAR_Sequence_Mapping_Readiness.csv',ROOT/'S2b_Human_NSC_Bait_Fragments_Embedded_for_Colab.csv']:
        z.write(file,arcname='inputs/'+file.name)
print('ZIP ready:',zip_path,'size:',zip_path.stat().st_size)
try:
    from google.colab import files as colab_files
    colab_files.download(str(zip_path))
except Exception as e:
    print('Not running in Colab? ZIP remains saved at:',zip_path)
